In [1]:
import heapq
from math import ceil
from typing import Iterable, Tuple, Dict, List, Hashable
from typing import TypeVar, Generic, Literal

import numpy as np
import polars as pl
from datasets import tqdm
from numpy.typing import NDArray
from ortools.graph.python.min_cost_flow import SimpleMinCostFlow

from libs.DataLoader import Loader
from libs.constants import *
from libs.utils import NDCG, count_polars

In [2]:
def get_weights(indices: NDArray, ranks: NDArray,
                method: Literal['mul', 'add'] = 'mul',
                temporal_distribution: Literal['linear', 'exp', 'uni'] = 'linear',
                alpha: float = 0.5, newest_first: bool = False) -> NDArray:
    """
    :param indices: array-like, temporal indices (only order matters)
    :param ranks: array-like, ranks magnitude (only magnitude matters)
    :param method: 'mul' or 'add'
    :param temporal_distribution: distribution of weights applied to temporal indices
    :param alpha: for 'add' method - weight of temporal component
    :param newest_first : if True lower index -> higher temporal-weight
    """
    indices = np.asarray(indices)
    ranks = np.asarray(ranks, dtype=float)
    if indices.shape != ranks.shape:
        raise NotImplementedError()

    n = len(indices)
    order = np.argsort(np.argsort(indices)) + 1
    if newest_first:
        order = n + 1 - order

    if temporal_distribution == 'linear':
        temporal_w = order / order.sum()
    elif temporal_distribution == 'exp':
        temporal_w = np.exp(order) / np.sum(np.exp(order))
    elif temporal_distribution == 'uni':
        temporal_w = np.ones(n) / n
    else:
        raise NotImplementedError()

    mag_sum = ranks.sum()
    if mag_sum == 0:
        magnitude_w = np.ones_like(ranks) / n
    else:
        magnitude_w = ranks / mag_sum

    if method == 'mul':
        combined = temporal_w * magnitude_w
    elif method == 'add':
        combined = alpha * temporal_w + (1.0 - alpha) * magnitude_w
    else:
        raise NotImplementedError()

    return combined / combined.sum()

In [3]:
U = TypeVar('U', bound=Hashable)
I = TypeVar('I', bound=Hashable)

class Solver(Generic[I, U]):
    """
    Two-phase solver:
      1) Stream scores from `score_generator` and keep top-L candidates per item.
      2) Solve reduced assignment with min-cost flow (OR-Tools) or greedy fallback.

    Parameters
    ----------
    L : int
        Number of candidates to keep per item (memory/quality knob).
    K : int
        Number of users to select per item.
    R : int
        Max times a user can be assigned across all items.
    use_ortools : bool
        If True, prefer OR-Tools solver when available; otherwise use greedy fallback.
    score_scale : float
        Scaling factor for converting float scores to integer costs for OR-Tools.
    """

    def __init__(self, L: int, K: int, R: int, use_ortools: bool = True, score_scale: float = 1e6):
        assert L > 0 and K > 0 and R > 0
        self.L = L
        self.K = K
        self.R = R
        self.use_ortools = use_ortools
        self.score_scale = score_scale

        # item -> dict(user -> best_score)
        self._item_to_user_score: Dict[I, Dict[U, float]] = {}
        # item -> heap of (score, user) used to evict smallest when > L
        self._item_heaps: Dict[I, List[Tuple[float, U]]] = {}

    def collect_candidates(self, score_gen: Iterable[Tuple[NDArray, list[U], list[I]]]):
        """
        Consume your score_generator batches and populate internal candidate lists.

        score_gen yields (batch_scores, users, items)
          - batch_scores: numpy array shape (len(users), len(items)) or (n_users, n_items_batch)
          - users: list of user ids corresponding to rows
          - items: list of item ids corresponding to columns
        """
        for batch_scores, users, items in score_gen:
            # iterate columns (items) to avoid iterating over full matrix
            for col_idx, item in enumerate(items):
                # lazy init per item
                mapping = self._item_to_user_score.get(item)
                if mapping is None:
                    mapping = {}
                    self._item_to_user_score[item] = mapping
                    self._item_heaps[item] = []

                heap = self._item_heaps[item]
                # extract column vector of scores
                # assume batch_scores is numpy-like with indexing [row, col]
                col_scores = batch_scores[:, col_idx]
                for row_idx, user in enumerate(users):
                    s = float(col_scores[row_idx])
                    # if user already present, possibly update
                    prev = mapping.get(user)
                    if prev is None:
                        # add if there's room
                        if len(mapping) < self.L:
                            mapping[user] = s
                            heapq.heappush(heap, (s, user))
                        else:
                            # If new score beats current min, add and evict
                            # heap[0] is smallest (min-heap)
                            if heap and s > heap[0][0]:
                                mapping[user] = s
                                heapq.heappush(heap, (s, user))
                                # Evict until mapping size <= L and heap top matches mapping
                                self._evict_until_within_L(item)
                    else:
                        # user already present: update if this score is larger
                        if s > prev:
                            mapping[user] = s
                            heapq.heappush(heap, (s, user))
                            # If mapping grew beyond L (shouldn't unless we inserted new user), evict
                            if len(mapping) > self.L:
                                self._evict_until_within_L(item)

        # After all batches, optionally prune heap leftovers (make mapping consistent)
        for item in list(self._item_to_user_score.keys()):
            self._cleanup_item_heap(item)

    def get_reduced_edges(self) -> List[Tuple[I, U, float]]:
        """
        Return list of (item, user, score) for current collected candidates.
        """
        edges = []
        for item, umap in self._item_to_user_score.items():
            for user, score in umap.items():
                edges.append((item, user, score))
        return edges

    def _evict_until_within_L(self, item: I):
        mapping = self._item_to_user_score[item]
        heap = self._item_heaps[item]
        # sanity: if heap empties but mapping still too large, rebuild a fresh heap from mapping
        while len(mapping) > self.L:
            if not heap:
                # rebuild heap from current mapping items
                heap = [(s, u) for u, s in mapping.items()]
                heapq.heapify(heap)
                self._item_heaps[item] = heap
                if not heap:
                    break  # nothing to pop
            s_pop, u_pop = heapq.heappop(heap)
            current = mapping.get(u_pop)
            if current is None:
                continue
            # use a tolerant comparison
            if abs(current - s_pop) < 1e-12:
                del mapping[u_pop]
            else:
                # stale entry, continue popping
                continue

    def _cleanup_item_heap(self, item: I):
        """Remove stale heap entries and ensure mapping and heap are consistent."""
        mapping = self._item_to_user_score[item]
        heap = self._item_heaps[item]
        new_heap = []
        for s, u in heap:
            # only keep entries that match mapping
            if u in mapping and abs(mapping[u] - s) < 1e-12:
                new_heap.append((s, u))
        heapq.heapify(new_heap)
        self._item_heaps[item] = new_heap
        # If mapping somehow exceeded L, evict down to L
        if len(mapping) > self.L:
            self._evict_until_within_L(item)

    def solve(self) -> Dict[I, List[Tuple[U, float]]]:
        """
        Solve assignment on the reduced candidate graph.

        Returns
        -------
        assignments : dict
          item_id -> list of (user_id, score), length K for each item (if feasible)
        """
        # Build candidate list edges: (item, user, score)
        edges = []
        for item, user_map in self._item_to_user_score.items():
            for user, score in user_map.items():
                edges.append((item, user, score))

        if not edges:
            return {}

        # Unique node mappings
        items = sorted({e[0] for e in edges})
        users = sorted({e[1] for e in edges})
        item_to_idx = {it: idx for idx, it in enumerate(items)}
        user_to_idx = {u: idx for idx, u in enumerate(users)}

        total_item_demand = len(items) * self.K
        total_user_capacity = len(users) * self.R
        if total_user_capacity < total_item_demand:
            raise RuntimeError(
                f"Impossible to satisfy all demands: total user capacity {total_user_capacity} < total item demand {total_item_demand}."
                " Increase R or collect more candidate users per item (increase L)."
            )

        # If OR-Tools is available and requested, use min-cost flow
        if self.use_ortools:
            return self._solve_with_ortools(items, users, edges, item_to_idx, user_to_idx)
        else:
            raise NotImplementedError()

    def _solve_with_ortools(self, items, users, edges, item_to_idx, user_to_idx):
        # Build min-cost flow graph with nodes:
        # source (0), item nodes 1..I, user nodes I+1..I+U, sink = I+U+1
        I = len(items)
        U = len(users)
        source = 0
        item_base = 1
        user_base = item_base + I
        sink = user_base + U

        min_cost_flow = SimpleMinCostFlow()

        # helper to add arc and record arc ids for later flow extraction
        arc_identifiers = []  # ((item_idx, user_idx), arc_index_in_or_tools)
        max_score = max(s for (_, _, s) in edges)
        # Add source -> item arcs (capacity K, cost 0)
        for i_idx in range(I):
            min_cost_flow.add_arc_with_capacity_and_unit_cost(source, item_base + i_idx, self.K, 0)

        # Add user -> sink arcs (capacity R, cost 0)
        for u_idx in range(U):
            min_cost_flow.add_arc_with_capacity_and_unit_cost(user_base + u_idx, sink, self.R, 0)

        # Add item->user arcs for every candidate (capacity 1, cost based on score)
        # To maximize score, set cost = int(round((max_score - score) * scale))
        for (item, user, score) in edges:
            i_idx = item_to_idx[item]
            u_idx = user_to_idx[user]
            # transform score -> non-negative integer cost (smaller cost = better score)
            # scale and round carefully. Ensure cost is int.
            cost = int(round((max_score - score) * self.score_scale))
            arc_id = min_cost_flow.add_arc_with_capacity_and_unit_cost(item_base + i_idx, user_base + u_idx, 1, cost)
            arc_identifiers.append(((i_idx, u_idx), arc_id))

        # Set supplies: source has +total demand, sink has -total demand
        total_demand = len(items) * self.K
        min_cost_flow.set_node_supply(source, total_demand)
        min_cost_flow.set_node_supply(sink, -total_demand)

        status = min_cost_flow.solve_max_flow_with_min_cost()
        if status != min_cost_flow.OPTIMAL:
            raise NotImplementedError()

        # Extract chosen edges (flow == 1 on item->user arcs)
        # Build reverse maps
        idx_to_item = {v: k for k, v in item_to_idx.items()}
        idx_to_user = {v: k for k, v in user_to_idx.items()}
        assignments = {item: [] for item in items}
        # For each arc, query flow
        for ((i_idx, u_idx), arc_id) in arc_identifiers:
            if min_cost_flow.flow(arc_id) > 0:
                item = idx_to_item[i_idx]
                user = idx_to_user[u_idx]
                score = self._item_to_user_score[item][user]
                assignments[item].append((user, score))

        # Ensure each list sorted by descending score and limited to K
        for it in assignments:
            assignments[it] = sorted(assignments[it], key=lambda x: -x[1])[:self.K]
        return assignments

In [4]:
def score_generator(data: pl.LazyFrame, val: pl.LazyFrame, items_df: pl.LazyFrame,
                    users_batch_size: int=1_000_000, items_batch_size: int=1_000_000
                    ) -> Iterable[Tuple[NDArray, list[int], list[int]]]:
    """
    :param data: LazyFrame with columns [ITEM, USER, TIME_INDEX, TARGET], where USER, TIME_INDEX, TARGET are List[n_user_interactions]
    :param val: LazyFrame with columns [ITEM] - data to predict
    :param items_df: LazyFrame with index [ITEM] and columns [EMBEDDING] - contains all items with their embeddings
    :param users_batch_size: batching for users
    :param items_batch_size: batching for items
    """
    n_data = count_polars(data)
    n_val = count_polars(val)

    data_bar = tqdm(total=ceil(n_data / users_batch_size), position=0, desc="data")
    val_bar = tqdm(total=ceil(n_val / items_batch_size), position=1, desc="val")

    data_bar.reset()
    for data_batch in data.collect_batches(chunk_size=users_batch_size):
        data_items_lists = data_batch[ITEM].to_list()  # list of lists
        time_index_lists = data_batch[TIME_INDEX].to_list()
        target_lists = data_batch[TARGET].to_list()
        users = data_batch[USER].to_list()

        seen = set()
        data_items_list = []
        user_item_weights = []
        for row_items, row_time, row_target in zip(data_items_lists, time_index_lists, target_lists):
            weights = get_weights(row_time, row_target, method="add", temporal_distribution="linear", alpha=0.5)
            items_weights = []
            for it, w in zip(row_items, weights):
                if it not in seen:
                    seen.add(it)
                    data_items_list.append(it)
                items_weights.append((it, float(w)))
            user_item_weights.append(items_weights)

        agg_embeddings = None
        for start in range(0, len(data_items_list), items_batch_size):
            chunk = data_items_list[start:start + items_batch_size]

            items_batch_df = items_df.filter(pl.col(ITEM).is_in(chunk)).select([ITEM, EMBEDDING]).collect()
            chunk_map = {r[0]: np.asarray(r[1], dtype=np.float64) for r in items_batch_df.rows()}

            if agg_embeddings is None:
                d = next(iter(chunk_map.values())).shape[0]
                agg_embeddings = np.zeros((len(users), d), dtype=np.float64)

            for ui, items_weights in enumerate(user_item_weights):
                acc = agg_embeddings[ui]
                for it, w in items_weights:
                    emb = chunk_map.get(it)
                    if emb is not None:
                        acc += w * emb

        agg_norms = np.linalg.norm(agg_embeddings, axis=1, keepdims=True)
        agg_norms[agg_norms == 0] = 1e-12

        val_bar.reset()
        items, similarity_blocks = [], []
        for val_batch in val.collect_batches(chunk_size=items_batch_size):
            val_items_batch = val_batch[ITEM].to_list()
            items.extend(val_items_batch)

            val_items_df_batch = items_df.filter(pl.col(ITEM).is_in(val_items_batch)).select([ITEM, EMBEDDING]).collect()
            val_item_to_emb = {r[0]: np.asarray(r[1], dtype=np.float64) for r in val_items_df_batch.rows()}

            val_embeddings = np.vstack([val_item_to_emb[it] for it in val_items_batch])  # (n_val_batch, d)
            val_norms = np.linalg.norm(val_embeddings, axis=1, keepdims=True)
            val_norms[val_norms == 0] = 1e-12

            similarity = (val_embeddings @ agg_embeddings.T) / (val_norms * agg_norms.T)  # (n_val_batch, n_users_batch)
            similarity_blocks.append(similarity.T)

            val_bar.update(1)
        data_bar.update(1)
        yield np.hstack(similarity_blocks), users, items

In [ ]:
loader = Loader('ur0.01', content_embedding_size=16, batch_size=500_000)
(train_df, val_df), users_df, items_df = loader.load_data(convert_to_pandas=False)

Load metadata


In [6]:
train_df: pl.LazyFrame = train_df.filter(pl.col(TARGET) > 0)  # Interaction: positive
val_df: pl.LazyFrame = val_df.filter(pl.col(TARGET) > 0)  # Interaction: positive

print(f"Train: {count_polars(train_df):_} Val: {count_polars(val_df):_}")

Train: 62_457 Val: 1_970


In [7]:
num_recent_videos = 100

agg = train_df \
    .sort(by=[pl.col(USER), pl.col(TIME_INDEX)], descending=[False, True]) \
    .group_by(USER) \
    .agg([
        pl.col(ITEM).slice(0, num_recent_videos).alias(ITEM),
        pl.col(TIME_INDEX).slice(0, num_recent_videos).alias(TIME_INDEX),
        pl.col(TARGET).slice(0, num_recent_videos).alias(TARGET)
    ])

val = val_df.select(ITEM).unique(ITEM)
cold_val = val.join(train_df.select(ITEM).unique(), on=ITEM, how="anti")

In [10]:
solver = Solver(500, 100, 101)
solver.collect_candidates(score_generator(agg, val, items_df, 500, 500))

data:   0%|          | 0/5 [00:00<?, ?it/s]

val:   0%|          | 0/3 [00:00<?, ?it/s]

In [11]:
results = solver.solve()

In [27]:
predict = pl.DataFrame({
    ITEM: list(results.keys()),
    USER: [[u for u, _ in users_scores] for users_scores in results.values()]
})

In [93]:
for pred, pred_name in [
    (predict, "constrained")
]:
    for true, true_name in [
        (val_df, "all"),
        (val_df.join(cold_val, on=ITEM, how='inner'), "cold")
    ]:
        pl.LazyFrame().group_by().agg()
        print(f"Predict: {pred_name} Test: {true_name} NDCG: {NDCG(pred.to_pandas(), true.group_by(ITEM).agg(pl.col(USER)).collect().to_pandas()):.4f}")

Predict: constrained Test: all NDCG: 0.0723
Predict: constrained Test: cold NDCG: 0.1061
